# Cut a processed STAFF III record

Load a Graph_Visualizer signal pair (`.npy` + `.pkl`), keep a time window, and write a new pair.

Default: STAFF III `039c`, **275–300 s**, saved as `39c_cut` under Graph_Visualizer `data/downloaded/staff_III/`.

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd() if (Path.cwd() / "evaluation" / "common.py").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO / "evaluation"))

import pickle

import numpy as np

import common as C

RECORD_ID = "039c"
T_START_S = 275.0
T_END_S = 300.0
OUT_STEM = "39c_cut"

THESIS = REPO.parent.parent
GV_DOWNLOADED = (
    THESIS / "Graph_Visualizer" / "graph-viewer" / "backend" / "data" / "downloaded" / "staff_III"
)
SOURCE_CANDIDATES = [
    REPO / "data" / "evaluation" / "processed" / "staff_iii" / "signals" / RECORD_ID,
    THESIS / "Graph_Visualizer" / "graph-viewer" / "backend" / "data" / "processed" / "staff_iii" / "signals" / RECORD_ID,
    THESIS / "Graph_Visualizer" / "graph-viewer" / "backend" / "data" / "uploaded_signals" / RECORD_ID,
]
DEST_STEM = GV_DOWNLOADED / OUT_STEM

src = next(
    (
        p
        for p in SOURCE_CANDIDATES
        if p.with_suffix(".npy").is_file() and p.with_suffix(".pkl").is_file()
    ),
    None,
)
if src is None:
    raise FileNotFoundError(f"No processed {RECORD_ID}.npy/.pkl in {SOURCE_CANDIDATES}")

print("source:", src.with_suffix(".npy"))
print("dest:", DEST_STEM.with_suffix(".npy"))

## Load, slice, save

In [ ]:
signal = np.load(src.with_suffix(".npy"))
with open(src.with_suffix(".pkl"), "rb") as handle:
    meta = pickle.load(handle)
fs = int(meta["fs"])
channels = list(meta["channels"])
n_samples = signal.shape[1]
duration_s = n_samples / fs
print(f"{RECORD_ID}: shape={signal.shape} fs={fs} duration={duration_s:.1f}s")

if T_END_S > duration_s:
    raise ValueError(f"T_END_S={T_END_S} exceeds recording length {duration_s:.1f}s")

i0 = int(round(T_START_S * fs))
i1 = int(round(T_END_S * fs))
cut = signal[:, i0:i1]
print(
    f"window [{T_START_S:g}, {T_END_S:g}) s → samples [{i0}, {i1}) "
    f"→ shape {cut.shape} ({cut.shape[1] / fs:.1f}s)"
)

npy_path, pkl_path = C.save_signal_pair(DEST_STEM, cut, fs, channels)
print("wrote", npy_path)
print("wrote", pkl_path)

## Check the cut

Interactive 12-lead of the saved window:

In [ ]:
cut_sig = np.load(npy_path)
with open(pkl_path, "rb") as handle:
    cut_meta = pickle.load(handle)
assert cut_sig.shape == cut.shape
assert int(cut_meta["fs"]) == fs
assert list(cut_meta["channels"]) == channels
print("reload ok", cut_sig.shape, cut_meta)

C.plot_12_lead(
    cut_sig,
    int(cut_meta["fs"]),
    list(cut_meta["channels"]),
    title=f"STAFF III {RECORD_ID} cut {T_START_S:.0f}–{T_END_S:.0f} s",
    show=True,
)